# 01 — Data download

Fetch 1-minute BTCUSDT klines and write a parquet that `wagie.io.sources.ParquetReplaySource` can ingest.

Schema expected by the source (see `wagie.engine.Engine` doctring): `open_time`, `open`, `high`, `low`, `close`, `volume`, `close_time`, `quote_volume`, `trades`, `taker_buy_base`, `taker_buy_quote`, `segment_id`. All numeric except `*_time` (epoch ms, integer).

## Option A — synthetic random walk (demos / CI)

If you don't have Binance access, generate a synthetic parquet with the canonical schema. Used by the contract suite to keep tests cold-startable.

In [ ]:
from pathlib import Path
import numpy as np
import polars as pl

out = Path("data/synthetic/btcusdt_1m.parquet")
out.parent.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
n = 60_000  # ~6 weeks of 1-min bars
sigma = 0.0008
log_ret = rng.normal(0.0, sigma, size=n); log_ret[0] = 0.0
close = np.exp(10.0 + np.cumsum(log_ret))
rng_h = np.abs(rng.normal(0.0, sigma * 1.2, size=n))
high = close * (1.0 + rng_h)
low  = close * (1.0 - rng_h)
open_ = np.r_[close[0], close[:-1]]
vol  = np.abs(rng.normal(100.0, 20.0, size=n))
base_ms = 1_700_000_000_000
open_time  = np.arange(n) * 60_000 + base_ms
close_time = open_time + 59_999

df = pl.DataFrame({
    "open_time": open_time, "open": open_, "high": high, "low": low,
    "close": close, "volume": vol, "close_time": close_time,
    "quote_volume": vol * close, "trades": np.full(n, 50, dtype=np.int64),
    "taker_buy_base":  vol * 0.5,
    "taker_buy_quote": vol * close * 0.5,
    "segment_id": np.zeros(n, dtype=np.int64),
})
df.write_parquet(out)
print(f"wrote {out}  rows={len(df)}")
df.head(5)

## Option B — real Binance klines

Public REST endpoint, no auth needed. Walk a date range, paginate at 1000 rows/request, stitch into one parquet. Set `segment_id = 0` unless you want to break apart over downtime windows.

In [ ]:
# Sketch — uncomment and tune for a real download.
#
# import requests, time
# from datetime import datetime, timezone
#
# def fetch_chunk(symbol: str, start_ms: int, end_ms: int, limit: int = 1000):
#     url = "https://api.binance.com/api/v3/klines"
#     r = requests.get(url, params={
#         "symbol": symbol, "interval": "1m",
#         "startTime": start_ms, "endTime": end_ms, "limit": limit,
#     })
#     r.raise_for_status()
#     return r.json()
#
# rows = []
# t0 = int(datetime(2024, 1, 1, tzinfo=timezone.utc).timestamp() * 1000)
# t1 = int(datetime(2024, 2, 1, tzinfo=timezone.utc).timestamp() * 1000)
# cur = t0
# while cur < t1:
#     chunk = fetch_chunk("BTCUSDT", cur, t1)
#     if not chunk: break
#     rows.extend(chunk)
#     cur = chunk[-1][6] + 1   # next open_time = last close_time + 1
#     time.sleep(0.25)         # be nice to the public endpoint
#
# # rows: [open_time, o, h, l, c, v, close_time, qv, trades, tb_base, tb_quote, _ignore]
# import polars as pl
# df = pl.DataFrame({
#     "open_time":  [int(r[0]) for r in rows],
#     "open":       [float(r[1]) for r in rows],
#     "high":       [float(r[2]) for r in rows],
#     "low":        [float(r[3]) for r in rows],
#     "close":      [float(r[4]) for r in rows],
#     "volume":     [float(r[5]) for r in rows],
#     "close_time": [int(r[6]) for r in rows],
#     "quote_volume":   [float(r[7]) for r in rows],
#     "trades":         [int(r[8]) for r in rows],
#     "taker_buy_base": [float(r[9]) for r in rows],
#     "taker_buy_quote":[float(r[10]) for r in rows],
#     "segment_id":     [0] * len(rows),
# })
# df.write_parquet("data/raw_data/BTCUSDT/btcusdt_1m_2024-01.parquet")

Once written, the parquet is ready to be referenced from a spec:

```yaml
wagie:
  data:
    parquet_path: data/synthetic/btcusdt_1m.parquet
    m_minutes: 20
```

Move on to **02_feature_build**.